In [1]:

from itertools import product
from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed, parallel_config

from deeprootgen.data_model import RootSimulationModel
from deeprootgen.model import RootSystemSimulation
from deeprootgen.statistics.root_grid_aggregation import aggregate_root_grid


# ============================================================
# 1. Configuration
# ============================================================

INFILE = Path("..", "..", "data", "grapevine", "fichtl_2024_voxel_grid.csv")
OUT_DIR = Path("outputs/grapevine_calibration")
OUT_DIR.mkdir(parents=True, exist_ok=True)

GROUPS_TO_FIT = [("101-14", "T1")]
# GROUPS_TO_FIT = None  # All 12 rootstock/timepoint groups

VOXEL_SIZE_CM = 10.0
BASE_SEED = 42
N_REPLICATES = 5
N_TEST_VINES = 2

# Process-based concurrency. Set to 1 to debug sequentially.
N_JOBS = min(6, os.cpu_count() or 1)

# If True, reuse completed candidates with the same configuration fingerprint.
RESUME_SEARCH = True
CHECKPOINT_EVERY = 10

# 'composite' preserves the objective in the user's original script.
# 'rld' calibrates only against balanced voxel-wise RLD and skips extra
# root-order aggregations, which can save substantial work.
OBJECTIVE_MODE = "composite"  # or "rld"

PARAMETER_GRID = {
    "max_primary_length": [25.0, 40.0],
    "outer_root_num": [3, 4, 5],
    "max_sec_root_num": [2, 3, 4],
    "max_sec_root_length": [8.0, 15.0],
    "interbranch_distance": [1.0, 2.0],
    "gravitropism": [0.05, 0.1],
}

LOSS_WEIGHTS = {"spatial": 0.60, "order": 0.25, "length": 0.15}
ORDER_COLS = {
    1: "primary_length_cm",
    2: "secondary_length_cm",
    3: "tertiary_length_cm",
    4: "quaternary_length_cm",
    5: "quinary_length_cm",
}
KEYS = ["x_min_cm", "y_min_cm", "depth_min_cm"]

if OBJECTIVE_MODE not in {"composite", "rld"}:
    raise ValueError("OBJECTIVE_MODE must be 'composite' or 'rld'.")
if N_JOBS < 1 or N_REPLICATES < 1 or CHECKPOINT_EVERY < 1:
    raise ValueError("N_JOBS, N_REPLICATES and CHECKPOINT_EVERY must be >= 1.")
if not np.isclose(sum(LOSS_WEIGHTS.values()), 1.0):
    raise ValueError("LOSS_WEIGHTS must sum to 1.")


# ============================================================
# 2. Baseline DeepRootGen model parameters
# ============================================================

BASE_PARAMETERS = dict(
    random_seed=BASE_SEED,
    max_order=3,
    root_ratio=0.5,
    fine_root_threshold=0.15,
    outer_root_num=4,
    inner_root_num=2,
    min_primary_length=15.0,
    max_primary_length=40.0,
    base_diameter=0.5,
    diameter_reduction=0.2,
    apex_diameter=0.05,
    min_sec_root_num=1,
    max_sec_root_num=3,
    growth_sec_root=0.1,
    min_sec_root_length=3.0,
    max_sec_root_length=15.0,
    segments_per_root=6,
    length_reduction=0.7,
    root_vary=10.0,
    interbranch_distance=2.0,
    mechanical_constraints=0.0,
    root_tissue_density=0.15,
    gravitropism=0.5,
    t=1,
    enable_soil=False,
    soil_layer_height=VOXEL_SIZE_CM,
    soil_n_layers=10,
    soil_layer_width=VOXEL_SIZE_CM,
    soil_n_cols=10,
)

# ============================================================
# 3. Load and validate the observations
# ============================================================

observed_df = pd.read_csv(INFILE)
required = [
    "rootstock", "timepoint", "vine_id", *KEYS,
    "root_length_cm", "rld_cm_per_cm3", *ORDER_COLS.values(),
    "other_length_cm",
]
missing = set(required).difference(observed_df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if observed_df[required].isna().any().any():
    raise ValueError("Missing observations in required columns.")
if (observed_df["depth_min_cm"] < 0).any():
    raise ValueError("Expected nonnegative observed depths.")
if (observed_df["other_length_cm"].abs() > 1e-9).any():
    raise ValueError("Nonzero 'other' root lengths need an explicit order mapping.")
if (observed_df["root_length_cm"] < 0).any():
    raise ValueError("Root lengths must be nonnegative.")

np.testing.assert_allclose(
    observed_df["root_length_cm"].to_numpy(float),
    observed_df["rld_cm_per_cm3"].to_numpy(float) * VOXEL_SIZE_CM**3,
    rtol=1e-5, atol=1e-5,
)
np.testing.assert_allclose(
    observed_df["root_length_cm"].to_numpy(float),
    observed_df[[*ORDER_COLS.values(), "other_length_cm"]].sum(axis=1).to_numpy(float),
    rtol=1e-5, atol=1e-5,
)

seed_index = {
    vine: i for i, vine in enumerate(sorted(observed_df["vine_id"].unique()))
}
print("Dataset validation passed.")
print(f"Observations: {len(observed_df):,}; vines: {observed_df['vine_id'].nunique()}")


# ============================================================
# 4. Define the observed 3D voxel grid for each vine
# ============================================================

def prepare_vine(vine_df):
    observed = vine_df.copy().reset_index(drop=True)
    if observed.duplicated(KEYS).any():
        raise ValueError("Duplicate observed voxel coordinates for one vine.")

    def edges_from_minima(column):
        starts = np.sort(observed[column].unique().astype(float))
        if len(starts) > 1 and not np.allclose(np.diff(starts), VOXEL_SIZE_CM):
            raise ValueError(f"Unexpected voxel spacing in {column}.")
        return np.r_[starts, starts[-1] + VOXEL_SIZE_CM]

    x_edges = edges_from_minima("x_min_cm")
    y_edges = edges_from_minima("y_min_cm")
    depth_edges = edges_from_minima("depth_min_cm")
    n_voxels = ((len(x_edges) - 1) * (len(y_edges) - 1)
                * (len(depth_edges) - 1))
    if len(observed) != n_voxels:
        raise ValueError("The vine's voxel grid is not a complete rectangular box.")

    # Observations have positive depth; DeepRootGen uses negative z.
    z_edges = np.sort(-depth_edges)
    soil_df = pd.MultiIndex.from_product(
        [x_edges, y_edges, z_edges], names=["x", "y", "z"]
    ).to_frame(index=False)
    return {"observed": observed, "soil": soil_df}


# ============================================================
# 5. Simulate and aggregate voxel RLD
# ============================================================

def simulate_predict(parameters, prepared, seed):
    params = RootSimulationModel(**{**parameters, "random_seed": int(seed)})
    simulation = RootSystemSimulation(random_seed=int(seed))
    simulation.run(params)
    node_df, edge_df = simulation.G.as_df()

    grid_df, totals_df = aggregate_root_grid(
        node_df=node_df,
        soil_df=prepared["soil"],
        length_source="geometric",
    )
    if grid_df["plant_id"].nunique() != 1:
        raise ValueError("Expected exactly one simulated plant.")
    grid_df = grid_df.reset_index(drop=True)

    predicted = grid_df[[
        "x_min", "y_min", "z_max", "root_length_cm", "rld_cm_cm3"
    ]].copy()
    predicted.columns = [
        "x_min_cm", "y_min_cm", "z_max", "sim_length_cm", "sim_rld"
    ]
    predicted["depth_min_cm"] = -predicted.pop("z_max")

    if OBJECTIVE_MODE == "composite":
        # Keep all parent nodes; mask only the non-matching child segments.
        for order, observed_col in ORDER_COLS.items():
            output_col = f"sim_{observed_col}"
            if order > int(parameters["max_order"]):
                predicted[output_col] = 0.0
                continue

            order_nodes = node_df.copy()
            if "invalid_root" in order_nodes.columns:
                already_invalid = order_nodes["invalid_root"].fillna(False).astype(bool)
            else:
                already_invalid = pd.Series(False, index=order_nodes.index)
            order_nodes["invalid_root"] = (
                already_invalid | order_nodes["order"].ne(order)
            )
            order_grid, _ = aggregate_root_grid(
                node_df=order_nodes,
                soil_df=prepared["soil"],
                length_source="geometric",
            )
            np.testing.assert_allclose(
                order_grid[["x_min", "y_min", "z_max"]].to_numpy(float),
                grid_df[["x_min", "y_min", "z_max"]].to_numpy(float),
                atol=1e-8,
            )
            predicted[output_col] = order_grid["root_length_cm"].to_numpy(float)

        np.testing.assert_allclose(
            predicted[[f"sim_{c}" for c in ORDER_COLS.values()]].sum(axis=1),
            predicted["sim_length_cm"],
            rtol=1e-6, atol=1e-6,
        )

    return predicted


# ============================================================
# 6. Compare predictions against observations
# ============================================================

def score_prediction(prepared, predicted):
    observed = prepared["observed"].copy()
    predicted = predicted.copy()
    for key in KEYS:
        observed[key] = observed[key].round(8)
        predicted[key] = predicted[key].round(8)

    comparison = observed.merge(
        predicted, on=KEYS, how="left", validate="one_to_one"
    )
    if len(comparison) != len(observed) or comparison["sim_rld"].isna().any():
        raise ValueError("Some observed voxels could not be matched.")

    actual = comparison["rld_cm_per_cm3"].to_numpy(float)
    estimate = comparison["sim_rld"].to_numpy(float)
    squared_error = (estimate - actual)**2
    occupied = actual > 0

    if occupied.any() and (~occupied).any():
        balanced_mse = 0.5 * (
            squared_error[occupied].mean() + squared_error[~occupied].mean()
        )
        zero_mse = 0.5 * np.mean(actual[occupied]**2)
    else:
        balanced_mse = squared_error.mean()
        zero_mse = np.mean(actual**2)
    spatial_loss = float(balanced_mse / max(zero_mse, 1e-12))

    observed_length = float(comparison["root_length_cm"].sum())
    simulated_length = float(comparison["sim_length_cm"].sum())
    length_loss = float(
        ((simulated_length - observed_length) / max(observed_length, 1e-9))**2
    )

    if OBJECTIVE_MODE == "composite":
        actual_orders = comparison[list(ORDER_COLS.values())].to_numpy(float)
        estimated_orders = comparison[
            [f"sim_{c}" for c in ORDER_COLS.values()]
        ].to_numpy(float)
        order_loss = float(
            np.sum((estimated_orders - actual_orders)**2)
            / max(np.sum(actual_orders**2), 1e-12)
        )
        objective = (
            LOSS_WEIGHTS["spatial"] * spatial_loss
            + LOSS_WEIGHTS["order"] * order_loss
            + LOSS_WEIGHTS["length"] * length_loss
        )
    else:
        order_loss = float("nan")  # Not calculated in RLD-only mode.
        objective = spatial_loss

    comparison["rld_residual"] = estimate - actual
    metrics = {
        "objective": float(objective),
        "rmse": float(np.sqrt(squared_error.mean())),
        "zero_rmse": float(np.sqrt(np.mean(actual**2))),
        "spatial_loss": spatial_loss,
        "order_loss": order_loss,
        "length_loss": length_loss,
        "observed_length_cm": observed_length,
        "simulated_length_cm": simulated_length,
        "observed_occupied": int(occupied.sum()),
        "simulated_occupied": int((estimate > 0).sum()),
        "overlap_voxels": int(np.sum(occupied & (estimate > 0))),
    }
    return comparison, metrics


# ============================================================
# 7. Candidate evaluator: vines and replicates within one worker
# ============================================================

def evaluate_candidate(parameters, prepared, vine_ids, save_comparisons=False,
                       test=False):
    records = []
    comparisons = []
    for vine_id in vine_ids:
        for replicate in range(N_REPLICATES):
            seed = (BASE_SEED + seed_index[vine_id] * 1000 + replicate
                    + (10_000_000 if test else 0))
            prediction = simulate_predict(parameters, prepared[vine_id], seed)
            comparison, metrics = score_prediction(prepared[vine_id], prediction)
            records.append({
                "vine_id": vine_id, "replicate": replicate,
                "seed": seed, **metrics,
            })
            if save_comparisons:
                comparison = comparison.copy()
                comparison["replicate"] = replicate
                comparison["seed"] = seed
                comparisons.append(comparison)
    return pd.DataFrame(records), comparisons


def evaluate_grid_combination(candidate_id, values, base, prepared,
                              train_vines, rootstock, timepoint):
    """One joblib task = one parameter combination (no nested joblib)."""
    candidate = dict(zip(PARAMETER_GRID, values))
    parameters = {**base, **candidate}
    scores, _ = evaluate_candidate(parameters, prepared, train_vines)
    vine_scores = scores.groupby("vine_id").mean(numeric_only=True)
    row = {
        "candidate_id": int(candidate_id),
        "rootstock": rootstock,
        "timepoint": timepoint,
        **candidate,
        "max_order": base["max_order"],
        "train_objective": float(vine_scores["objective"].mean()),
        "train_rmse": float(vine_scores["rmse"].mean()),
        "train_spatial_loss": float(vine_scores["spatial_loss"].mean()),
        "train_length_loss": float(vine_scores["length_loss"].mean()),
        "train_objective_sd": float(scores["objective"].std(ddof=1)),
        "n_training_simulations": int(len(scores)),
    }
    if OBJECTIVE_MODE == "composite":
        row["train_order_loss"] = float(vine_scores["order_loss"].mean())
    return row


# ============================================================
# 8. Checkpointed, process-parallel grid search for one group
# ============================================================

def save_checkpoint(rows, path):
    frame = pd.DataFrame(rows).drop_duplicates("candidate_id", keep="last")
    frame = frame.sort_values("candidate_id")
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp_path, index=False)
    os.replace(tmp_path, path)


def calibrate_group(rootstock, timepoint):
    group = observed_df.loc[
        observed_df["rootstock"].eq(rootstock)
        & observed_df["timepoint"].eq(timepoint)
    ]
    if group.empty:
        raise ValueError(f"No observations for {rootstock}/{timepoint}.")

    prepared = {
        vine_id: prepare_vine(vine_df)
        for vine_id, vine_df in group.groupby("vine_id")
    }
    vines = np.array(sorted(prepared))
    if not 0 < N_TEST_VINES < len(vines):
        raise ValueError("N_TEST_VINES must be in [1, n_vines - 1].")
    shuffled = np.random.default_rng(BASE_SEED).permutation(vines)
    test_vines = sorted(shuffled[:N_TEST_VINES].tolist())
    train_vines = sorted(shuffled[N_TEST_VINES:].tolist())

    # Determine order from training observations only.
    train_obs = group.loc[group["vine_id"].isin(train_vines)]
    orders_present = [
        order for order, col in ORDER_COLS.items() if train_obs[col].sum() > 0
    ]
    max_order = max([int(BASE_PARAMETERS["max_order"]), *orders_present])
    base = {**BASE_PARAMETERS, "max_order": max_order}

    combinations = list(product(*PARAMETER_GRID.values()))
    n_simulations = len(combinations) * len(train_vines) * N_REPLICATES
    prefix = f"{rootstock.replace('-', '_')}_{timepoint}"

    # Signature prevents accidentally reusing checkpoints after changing
    # parameters, data, model settings, objective or train/test split.
    file_info = INFILE.stat()
    signature_payload = {
        "rootstock": rootstock, "timepoint": timepoint,
        "grid": PARAMETER_GRID, "base": base,
        "train_vines": train_vines, "test_vines": test_vines,
        "n_replicates": N_REPLICATES, "base_seed": BASE_SEED,
        "voxel_size_cm": VOXEL_SIZE_CM,
        "mode": OBJECTIVE_MODE, "weights": LOSS_WEIGHTS,
        "data_path": str(INFILE.resolve()),
        "data_size": file_info.st_size, "data_modified_ns": file_info.st_mtime_ns,
    }
    signature = hashlib.sha256(
        json.dumps(signature_payload, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    checkpoint = OUT_DIR / f"{prefix}_checkpoint_{signature}.csv"

    if RESUME_SEARCH and checkpoint.exists():
        completed_rows = pd.read_csv(checkpoint).to_dict("records")
    else:
        completed_rows = []

    completed_ids = {int(row["candidate_id"]) for row in completed_rows}
    pending = [
        (i, values) for i, values in enumerate(combinations)
        if i not in completed_ids
    ]

    print(f"\nCALIBRATING {rootstock} / {timepoint}")
    print(f"Training vines: {train_vines}")
    print(f"Held-out vines: {test_vines}")
    print(f"Objective: {OBJECTIVE_MODE}")
    print(f"Parameter combinations: {len(combinations):,}")
    print(f"Training simulations: {n_simulations:,}")
    print(f"Workers: {N_JOBS}; replicates per vine: {N_REPLICATES}")
    print(f"Completed combinations found: {len(completed_ids)}")
    print(f"Checkpoint: {checkpoint}")

    if pending:
        with parallel_config(
            backend="loky", n_jobs=N_JOBS, inner_max_num_threads=1,
        ):
            # The unordered generator supplies finished rows without waiting
            # for every task; write checkpoints periodically.
            result_stream = Parallel(
                return_as="generator_unordered",
                verbose=5,
                pre_dispatch="2*n_jobs",
                batch_size=1,
            )(
                delayed(evaluate_grid_combination)(
                    candidate_id=i,
                    values=values,
                    base=base,
                    prepared=prepared,
                    train_vines=train_vines,
                    rootstock=rootstock,
                    timepoint=timepoint,
                )
                for i, values in pending
            )
            for j, row in enumerate(result_stream, start=1):
                completed_rows.append(row)
                if j % CHECKPOINT_EVERY == 0:
                    save_checkpoint(completed_rows, checkpoint)
                    print(
                        f"Saved: {len(completed_rows)}/{len(combinations)} "
                        f"combinations; latest objective="
                        f"{row['train_objective']:.5f}"
                    )
        save_checkpoint(completed_rows, checkpoint)

    # Sort by performance; preserve deterministic tie-breaking.
    search_df = (
        pd.DataFrame(completed_rows)
        .drop_duplicates("candidate_id", keep="last")
        .sort_values(["train_objective", "candidate_id"])
        .reset_index(drop=True)
    )
    if len(search_df) != len(combinations):
        raise RuntimeError("Grid search ended before every candidate was evaluated.")
    best_row = search_df.iloc[0]
    best_choice = {
        key: (best_row[key].item() if isinstance(best_row[key], np.generic)
              else best_row[key])
        for key in PARAMETER_GRID
    }
    best_parameters = {**base, **best_choice}

    # Re-run only the winning parameters to save individual training scores.
    # Using identical seeds makes this reproducible if the simulator is seeded.
    train_scores, _ = evaluate_candidate(
        best_parameters, prepared, train_vines, save_comparisons=False
    )
    # Fresh, deterministic random seeds for held-out evaluation.
    test_scores, comparisons = evaluate_candidate(
        best_parameters, prepared, test_vines,
        save_comparisons=True, test=True,
    )
    test_comparison = pd.concat(comparisons, ignore_index=True)

    search_df.to_csv(OUT_DIR / f"{prefix}_grid_search.csv", index=False)
    train_scores.to_csv(OUT_DIR / f"{prefix}_train_scores.csv", index=False)
    test_scores.to_csv(OUT_DIR / f"{prefix}_test_scores.csv", index=False)
    test_comparison.to_csv(
        OUT_DIR / f"{prefix}_test_voxel_comparison.csv", index=False
    )
    pd.DataFrame([best_parameters]).to_csv(
        OUT_DIR / f"{prefix}_best_parameters.csv", index=False
    )
    with open(OUT_DIR / f"{prefix}_run_config.json", "w", encoding="utf-8") as fh:
        json.dump(
            {**signature_payload, "signature": signature,
             "n_jobs": N_JOBS, "best_parameters": best_parameters},
            fh, indent=2,
        )

    print("\nBEST PARAMETERS")
    print(best_choice)
    print("Mean training objective:", train_scores["objective"].mean())
    print("Mean held-out objective:", test_scores["objective"].mean())
    print("Mean held-out RMSE:", test_scores["rmse"].mean())
    print("\nTOP GRID-SEARCH RESULTS")
    display(search_df.head(15))
    print("\nHELD-OUT VINE RESULTS")
    display(test_scores.groupby("vine_id")[[
        "objective", "rmse", "overlap_voxels"
    ]].mean())

    return {
        "search": search_df,
        "best_parameters": best_parameters,
        "train_scores": train_scores,
        "test_scores": test_scores,
        "test_comparison": test_comparison,
    }


# ============================================================
# 9. Execute calibration
# ============================================================

if GROUPS_TO_FIT is None:
    groups_to_fit = list(
        observed_df[["rootstock", "timepoint"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
else:
    groups_to_fit = GROUPS_TO_FIT

calibration_results = {}
for rootstock, timepoint in groups_to_fit:
    calibration_results[(rootstock, timepoint)] = calibrate_group(
        rootstock, timepoint
    )

print("\nCalibration complete. Outputs:", OUT_DIR.resolve())


Dataset validation passed.
Observations: 63,653; vines: 96

CALIBRATING 101-14 / T1
Training vines: ['B-23', 'B-24', 'B-45', 'B-57', 'B-58', 'B-76']
Held-out vines: ['B-46', 'B-52']
Objective: composite
Parameter combinations: 144
Training simulations: 4,320
Workers: 6; replicates per vine: 5
Completed combinations found: 0
Checkpoint: outputs/grapevine_calibration/101_14_T1_checkpoint_491a814d54d3.csv


[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:   41.4s


Saved: 10/144 combinations; latest objective=0.80975
Saved: 20/144 combinations; latest objective=0.78664
Saved: 30/144 combinations; latest objective=0.76426
Saved: 40/144 combinations; latest objective=0.74325
Saved: 50/144 combinations; latest objective=0.73395


[Parallel(n_jobs=6)]: Done  60 tasks      | elapsed:  2.2min


Saved: 60/144 combinations; latest objective=0.73983
Saved: 70/144 combinations; latest objective=0.70705
Saved: 80/144 combinations; latest objective=0.81991
Saved: 90/144 combinations; latest objective=0.81107
Saved: 100/144 combinations; latest objective=0.78132
Saved: 110/144 combinations; latest objective=0.76735
Saved: 120/144 combinations; latest objective=0.76299
Saved: 130/144 combinations; latest objective=0.75527
Saved: 140/144 combinations; latest objective=0.75037


[Parallel(n_jobs=6)]: Done 144 out of 144 | elapsed:  4.8min finished



BEST PARAMETERS
{'max_primary_length': 25.0, 'outer_root_num': 5, 'max_sec_root_num': 4, 'max_sec_root_length': 15.0, 'interbranch_distance': 1.0, 'gravitropism': 0.1}
Mean training objective: 0.7070460794632728
Mean held-out objective: 0.6823235945465564
Mean held-out RMSE: 0.01648506877305142

TOP GRID-SEARCH RESULTS


,candidate_id,rootstock,timepoint,max_primary_length,outer_root_num,max_sec_root_num,max_sec_root_length,interbranch_distance,gravitropism,max_order,train_objective,train_rmse,train_spatial_loss,train_length_loss,train_objective_sd,n_training_simulations,train_order_loss
0,69,101-14,T1,25.0,5,4,15.0,1.0,0.10,3,0.707046,0.019813,0.711803,0.331007,0.051546,30,0.921252
1,71,101-14,T1,25.0,5,4,15.0,2.0,0.10,3,0.707046,0.019813,0.711803,0.331007,0.051546,30,0.921252
2,68,101-14,T1,25.0,5,4,15.0,1.0,0.05,3,0.707439,0.019798,0.712622,0.333731,0.053492,30,0.919226
3,70,101-14,T1,25.0,5,4,15.0,2.0,0.05,3,0.707439,0.019798,0.712622,0.333731,0.053492,30,0.919226
4,61,101-14,T1,25.0,5,3,15.0,1.0,0.10,3,0.719777,0.019723,0.710301,0.458757,0.044255,30,0.899134
5,63,101-14,T1,25.0,5,3,15.0,2.0,0.10,3,0.719777,0.019723,0.710301,0.458757,0.044255,30,0.899134
6,60,101-14,T1,25.0,5,3,15.0,1.0,0.05,3,0.721015,0.019723,0.711815,0.460542,0.043974,30,0.899377
7,62,101-14,T1,25.0,5,3,15.0,2.0,0.05,3,0.721015,0.019723,0.711815,0.460542,0.043974,30,0.899377
8,64,101-14,T1,25.0,5,4,8.0,1.0,0.05,3,0.724618,0.019796,0.721262,0.449690,0.042166,30,0.897630
9,66,101-14,T1,25.0,5,4,8.0,2.0,0.05,3,0.724618,0.019796,0.721262,0.449690,0.042166,30,0.897630



HELD-OUT VINE RESULTS


,objective,rmse,overlap_voxels
vine_id,,,
B-46,0.653175,0.015404,19.8
B-52,0.711472,0.017567,21.8



Calibration complete. Outputs: /mnt/c/Users/james/projects/deep-root-gen/notebooks/grapevine/outputs/grapevine_calibration
